# Week 9 Workshop: Visualization Principles

## Coding Visualizations With Real Budget Execution Data

In this workshop, you will build five charts **in code** using the real `EJECUCION_PRESUPUESTAL.csv` dataset.
Each chart should answer a concrete analytical question, not just display a chart type.


## Setup

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Create figures directory for exports
os.makedirs('figures', exist_ok=True)

print("Libraries loaded successfully!")

In [ ]:
# Configure matplotlib for publication quality
plt.style.use('seaborn-v0_8-whitegrid')

# Custom settings
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 11
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10

print("Matplotlib configured for publication quality!")

In [ ]:
# Load the dataset
# File location: ../data/EJECUCION_PRESUPUESTAL.csv
df = pd.read_csv('../data/EJECUCION_PRESUPUESTAL.csv')

# Display basic information
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

In [ ]:
# Preview the data
df.head()

In [ ]:
# Check data types
df.info()

In [ ]:
# Prepare real aggregated datasets for visualization

# Clean text columns that contain non-breaking spaces
text_cols = ['Fuente de Financiación', 'Situación de Fondos', 'Recursos´Presupuestales',
             'Nombre Sector', 'Nombre Entidad', 'Nombre Nivel Uno Rubro']
for col in text_cols:
    df[col] = df[col].astype(str).str.replace('\xa0', ' ', regex=False).str.strip()

# Convert monetary columns to numeric
money_cols = ['Apropiación Vigente', 'Compromisos', 'Obligaciones', 'Pagos']
for col in money_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 1) Top sectors by appropriation
sector_summary = (
    df.groupby('Nombre Sector', as_index=False)
      .agg({
          'Apropiación Vigente': 'sum',
          'Compromisos': 'sum',
          'Obligaciones': 'sum',
          'Pagos': 'sum'
      })
)
sector_summary['Execution_Pct'] = (
    sector_summary['Compromisos'] / sector_summary['Apropiación Vigente'] * 100
).round(1)
sector_summary = sector_summary.sort_values('Apropiación Vigente', ascending=False)

# 2) Top entities for scatterplot analysis
entity_summary = (
    df.groupby('Nombre Entidad', as_index=False)
      .agg({
          'Apropiación Vigente': 'sum',
          'Compromisos': 'sum',
          'Pagos': 'sum'
      })
)
entity_summary['Execution_Pct'] = (
    entity_summary['Compromisos'] / entity_summary['Apropiación Vigente'] * 100
).round(1)
entity_summary = entity_summary[entity_summary['Apropiación Vigente'] > 0]
entity_summary = entity_summary.sort_values('Apropiación Vigente', ascending=False).head(20)

# 3) Composition by sector and top-level budget category
composition = (
    df.groupby(['Nombre Sector', 'Nombre Nivel Uno Rubro'], as_index=False)['Apropiación Vigente']
      .sum()
)
top_sectors = sector_summary.head(6)['Nombre Sector']
composition = composition[composition['Nombre Sector'].isin(top_sectors)]
composition = composition.pivot(index='Nombre Sector', columns='Nombre Nivel Uno Rubro', values='Apropiación Vigente').fillna(0)
composition = composition.div(composition.sum(axis=1), axis=0).mul(100).round(1)
composition = composition.reset_index()

# 4) Funding source and fund situation summary
source_summary = (
    df.groupby(['Fuente de Financiación', 'Situación de Fondos'], as_index=False)
      .agg({
          'Apropiación Vigente': 'sum',
          'Compromisos': 'sum'
      })
)
source_summary['Execution_Pct'] = (
    source_summary['Compromisos'] / source_summary['Apropiación Vigente'] * 100
).round(1)

print('Real aggregated datasets prepared!')
print(f"  - sector_summary: {sector_summary.shape}")
print(f"  - entity_summary: {entity_summary.shape}")
print(f"  - composition: {composition.shape}")
print(f"  - source_summary: {source_summary.shape}")

---

# Part 1: Data Preparation and Color Palette (15 minutes)

---

## Task 1.1: Explore the Data

In [ ]:
# Explore the sector summary
sector_summary.head(10)

In [ ]:
# Summary statistics for the main monetary variables
df[['Apropiación Vigente', 'Compromisos', 'Obligaciones', 'Pagos']].describe().T

## Task 1.2: Define Your Color Palette

In [ ]:
# TODO: Define your professional color palette
# Choose colors that work well together and serve different purposes

# YOUR CODE HERE
COLORS = {
    'primary': ___,      # Main data color (e.g., '#2C3E50', 'steelblue')
    'secondary': ___,    # Comparison data (e.g., '#3498DB', 'lightsteelblue')
    'accent': ___,       # Highlighting important values (e.g., '#1ABC9C')
    'warning': ___,      # Below target / negative (e.g., '#E74C3C', 'coral')
    'neutral': ___,      # Reference lines, text (e.g., '#95A5A6', 'gray')
}

# Sequential palette for stacked charts (light to dark)
SEQUENTIAL = [___, ___, ___, ___]  # e.g., ['#D5E8D4', '#97D077', '#5FAD41', '#2E7D32']

print("Color palette defined!")
print(f"Primary: {COLORS['primary']}")
print(f"Secondary: {COLORS['secondary']}")
print(f"Accent: {COLORS['accent']}")
print(f"Warning: {COLORS['warning']}")
print(f"Neutral: {COLORS['neutral']}")

In [ ]:
# Visualize your palette
fig, ax = plt.subplots(figsize=(10, 2))

# Show main colors
for i, (name, color) in enumerate(COLORS.items()):
    ax.add_patch(plt.Rectangle((i, 0), 1, 1, facecolor=color))
    ax.text(i + 0.5, -0.2, name, ha='center', va='top', fontsize=9)
    ax.text(i + 0.5, 0.5, color, ha='center', va='center', fontsize=8, 
            color='white' if name in ['primary', 'warning'] else 'black')

ax.set_xlim(0, len(COLORS))
ax.set_ylim(-0.5, 1)
ax.axis('off')
ax.set_title('My Color Palette', fontsize=12, loc='left')
plt.tight_layout()
plt.show()

---

# Part 2: Create 5 Chart Types (75 minutes)

---

## Chart 1: Horizontal Bar Chart - Which sectors concentrate the most budget? (15 minutes)

Create a clean horizontal bar chart showing the **top 10 sectors by `Apropiación Vigente`**.

Requirements:
- Sort values from largest to smallest
- Use one main color
- Add value labels at the end of each bar
- Write a title that states the takeaway


In [ ]:
# TODO: Create horizontal bar chart

fig, ax = plt.subplots(figsize=(10, 6))

# Keep only the top 10 sectors and sort for a horizontal bar chart
sorted_data = sector_summary.head(10).sort_values('Apropiación Vigente', ascending=True)

# YOUR CODE HERE
# Create horizontal bars
bars = ax.barh(
    ___,  # y: sector names
    ___,  # width: Apropiación Vigente
    color=___,  # Use COLORS['primary']
    edgecolor='none'
)

# Add value labels
for bar, value in zip(bars, sorted_data['Apropiación Vigente']):
    ax.text(
        bar.get_width() + 5,
        bar.get_y() + bar.get_height()/2,
        f'${value/1e9:.1f}B',
        va='center',
        fontsize=10
    )

# Remove unnecessary elements
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.set_xticks([])

# Title and labels
ax.set_title('Top sectors by current appropriation', fontsize=12, loc='left', pad=10)

# Add source
ax.text(0, -0.12, 'Source: datos.gov.co - Budget Execution Data', 
        transform=ax.transAxes, fontsize=8, color='gray')

plt.tight_layout()
plt.show()

In [ ]:
# TODO: Export Chart 1
# Uncomment and run after your chart looks good

# fig.savefig('figures/chart1_top_sectors_budget.png', dpi=300, bbox_inches='tight', facecolor='white')
# fig.savefig('figures/chart1_top_sectors_budget.svg', format='svg', bbox_inches='tight', facecolor='white')


## Chart 2: Grouped Bar Chart - Where is the largest gap between appropriation and commitments? (15 minutes)

Create a grouped bar chart comparing `Apropiación Vigente` vs `Compromisos` for the **top 8 sectors by appropriation**.

**Requirements:**
- Two bars per sector (appropriation and commitments)
- Use primary and secondary colors
- Add legend
- Show execution rate above each pair

In [ ]:
# TODO: Create grouped bar chart

fig, ax = plt.subplots(figsize=(12, 6))

# Set up bar positions
top_compare = sector_summary.head(8).copy()
x = np.arange(len(top_compare))
width = 0.35

# YOUR CODE HERE
# Create bars for approved budget
bars1 = ax.bar(
    x - width/2,
    ___,  # Apropiación Vigente values
    width,
    label='Approved',
    color=___  # Use COLORS['secondary']
)

# Create bars for executed budget
bars2 = ax.bar(
    x + width/2,
    ___,  # Compromisos values
    width,
    label='Executed',
    color=___  # Use COLORS['primary']
)

# Add execution rate labels above each pair
for i, (approved, executed, rate) in enumerate(zip(
    top_compare['Apropiación Vigente'],
    top_compare['Compromisos'],
    top_compare['Execution_Pct']
)):
    ax.text(i, max(approved, executed) + 10, f'{rate}%',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

# Customize
ax.set_xticks(x)
ax.set_xticklabels(top_compare['Nombre Sector'], rotation=45, ha='right')
ax.set_ylabel('Amount')
ax.set_title('Appropriation vs commitments by sector', fontsize=12, loc='left', pad=10)
ax.legend(frameon=False)

# Remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add light horizontal gridlines
ax.yaxis.grid(True, linestyle='-', linewidth=0.5, color='lightgray', alpha=0.7)
ax.set_axisbelow(True)

# Add source
ax.text(0, -0.15, 'Source: datos.gov.co - Budget Execution Data', 
        transform=ax.transAxes, fontsize=8, color='gray')

plt.tight_layout()
plt.show()

In [ ]:
# TODO: Export Chart 2
# Uncomment and run after your chart looks good

# fig.savefig('figures/chart2_sector_gap_comparison.png', dpi=300, bbox_inches='tight', facecolor='white')
# fig.savefig('figures/chart2_sector_gap_comparison.svg', format='svg', bbox_inches='tight', facecolor='white')


## Chart 3: Scatter Plot - Which entities combine large budgets with low execution? (15 minutes)

Create a scatter plot using `entity_summary`.

**Requirements:**
- X axis: `Apropiación Vigente`
- Y axis: `Execution_Pct`
- Highlight at least 2 outliers or interesting entities
- Add a reference line at 90%
- Use point size or color meaningfully

In [ ]:
# TODO: Create scatter plot

fig, ax = plt.subplots(figsize=(12, 5))

# YOUR CODE HERE

# Suggested subset: top 20 entities by appropriation
plot_data = entity_summary.copy()

# Main scatter
ax.scatter(
    ___,  # x: appropriation
    ___,  # y: execution percentage
    s=___,  # try scaling by Compromisos or Pagos
    color=___,
    alpha=0.7
)

# Reference line at 90%
ax.axhline(y=90, color=COLORS['neutral'], linestyle='--', linewidth=1.5)

# Minimal gridlines (horizontal only)
ax.yaxis.grid(True, linestyle='-', linewidth=0.5, color='lightgray', alpha=0.7)
ax.set_axisbelow(True)

# Labels and title
ax.set_xlabel('Apropiación Vigente')
ax.set_ylabel('Execution Rate (%)')
ax.set_title('Large budgets do not always mean high execution', fontsize=12, loc='left', pad=10)

# Annotate 2 interesting entities (example: high budget with lower execution)
# YOUR CODE HERE

# Add source
ax.text(0, -0.12, 'Source: datos.gov.co - Budget Execution Data', 
        transform=ax.transAxes, fontsize=8, color='gray')

plt.tight_layout()
plt.show()

In [ ]:
# TODO: Export Chart 3
# Uncomment and run after your chart looks good

# fig.savefig('figures/chart3_entity_execution_scatter.png', dpi=300, bbox_inches='tight', facecolor='white')
# fig.savefig('figures/chart3_entity_execution_scatter.svg', format='svg', bbox_inches='tight', facecolor='white')


## Chart 4: Stacked Bar Chart - How does budget composition differ across sectors? (15 minutes)

Create a stacked bar chart showing the composition of top sectors by `Nombre Nivel Uno Rubro`.

**Requirements:**
- Use the `composition` table prepared from the real dataset
- Sequential color palette (light to dark)
- Percentage labels within segments when readable
- Legend
- Clean styling

In [ ]:
# TODO: Create stacked bar chart

fig, ax = plt.subplots(figsize=(10, 6))

# Prepare data
sector_names = composition['Nombre Sector']
components = [col for col in composition.columns if col != 'Nombre Sector']

# YOUR CODE HERE

# Create stacked bars
bottom = np.zeros(len(sector_names))

for i, component in enumerate(components):
    values = composition[component]
    ax.bar(
        sector_names,
        values,
        bottom=bottom,
        label=component.replace('_', ' '),
        color=___,  # Use SEQUENTIAL[i]
        edgecolor='white',
        linewidth=0.5
    )
    
    # Add percentage labels in center of each segment
    for j, value in enumerate(values):
        if value > 5:  # Only label if segment is big enough
            ax.text(
                j,
                bottom[j] + value/2,
                f'{value}%',
                ha='center',
                va='center',
                fontsize=9,
                color='white' if i > 1 else 'black'
            )
    
    bottom += values

# Customize
ax.set_ylabel('Percentage of Total Budget')
ax.set_title('Budget composition by top-level rubro and sector', fontsize=12, loc='left', pad=10)
ax.legend(frameon=False, loc='upper right', bbox_to_anchor=(1.15, 1))

# Set y-axis limit to 100
ax.set_ylim(0, 100)

# Remove spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add source
ax.text(0, -0.12, 'Source: datos.gov.co - Budget Execution Data', 
        transform=ax.transAxes, fontsize=8, color='gray')

plt.tight_layout()
plt.show()

In [ ]:
# TODO: Export Chart 4
# Uncomment and run after your chart looks good

# fig.savefig('figures/chart4_sector_composition.png', dpi=300, bbox_inches='tight', facecolor='white')
# fig.savefig('figures/chart4_sector_composition.svg', format='svg', bbox_inches='tight', facecolor='white')


## Chart 5: Small Multiples - Which sectors are below the 90% execution target? (15 minutes)

Create a small multiples chart using the **top 8 sectors by appropriation**.

**Requirements:**
- Consistent scales across all subplots
- One subplot per sector
- Highlight sectors below the 90% target
- Reference line at 90%
- Minimal styling

In [ ]:
# TODO: Create small multiples

# Use the top 8 sectors by appropriation
plot_data = sector_summary.head(8).copy()

# Calculate grid dimensions
n_depts = len(plot_data)
n_cols = 4
n_rows = (n_depts + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 6))
axes = axes.flatten()

# Target threshold
target = 90

# YOUR CODE HERE

for i, (idx, row) in enumerate(plot_data.iterrows()):
    ax = axes[i]
    
    # Choose color based on whether above/below target
    bar_color = ___ if row['Execution_Pct'] >= target else ___  # primary vs warning
    
    # Create single bar
    ax.barh([0], [row['Execution_Pct']], color=bar_color, height=0.5)
    
    # Add target reference line
    ax.axvline(x=target, color=COLORS['neutral'], linestyle='--', linewidth=1)
    
    # Add value label
    ax.text(row['Execution_Pct'] + 1, 0, f"{row['Execution_Pct']:.1f}%",
            va='center', fontsize=9, fontweight='bold')
    
    # Department name as title
    ax.set_title(row['Nombre Sector'], fontsize=10, loc='left', pad=5)
    
    # Consistent x-axis
    ax.set_xlim(0, 105)
    ax.set_ylim(-0.5, 0.5)
    
    # Remove y-axis and simplify
    ax.set_yticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)

# Hide empty subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

# Add overall title
fig.suptitle('Execution by sector (Target: 90%)', fontsize=12, y=1.02)

plt.tight_layout()
plt.show()

In [ ]:
# TODO: Export Chart 5
# Uncomment and run after your chart looks good

# fig.savefig('figures/chart5_sector_small_multiples.png', dpi=300, bbox_inches='tight', facecolor='white')
# fig.savefig('figures/chart5_sector_small_multiples.svg', format='svg', bbox_inches='tight', facecolor='white')


---

# Part 3: Publication-Ready Formatting (20 minutes)

---

## Checklist: Review Your Charts

For each chart, verify:

| Requirement | Chart 1 | Chart 2 | Chart 3 | Chart 4 | Chart 5 |
|-------------|---------|---------|---------|---------|----------|
| Descriptive title | [ ] | [ ] | [ ] | [ ] | [ ] |
| Clear axis labels | [ ] | [ ] | [ ] | [ ] | [ ] |
| Consistent font sizes | [ ] | [ ] | [ ] | [ ] | [ ] |
| Color from palette | [ ] | [ ] | [ ] | [ ] | [ ] |
| Appropriate legend | [ ] | [ ] | [ ] | [ ] | [ ] |
| No unnecessary gridlines | [ ] | [ ] | [ ] | [ ] | [ ] |
| Source attribution | [ ] | [ ] | [ ] | [ ] | [ ] |

In [ ]:
# List exported files
import os

if os.path.exists('figures'):
    files = os.listdir('figures')
    print(f"Exported files ({len(files)}):")
    for f in sorted(files):
        size = os.path.getsize(f'figures/{f}') / 1024
        print(f"  - {f} ({size:.1f} KB)")
else:
    print("No figures exported yet.")

---

# Part 4: Critical Analysis (10 minutes)

---

## Critical Analysis Questions

### Question 1: Most Effective Chart Type

Which chart type was most effective for communicating the most important pattern in this dataset? Why?

_Your answer:_

---

### Question 2: Insight from Multiple Charts

What insight would be missed if you only showed the top sectors by appropriation?

_Your answer:_

---

### Question 3: One Chart for Decision-Maker

If you could only show one chart to a decision-maker, which would you choose and what question would it answer best?

_Your answer:_

---

### Question 4: Additional Data

What additional variable or time dimension would make these visualizations more analytically powerful?

_Your answer:_

---

## Color Palette Documentation

Document your color choices:

| Purpose | Hex Code | Rationale |
|---------|----------|----------|
| Primary | | |
| Secondary | | |
| Accent | | |
| Warning | | |
| Neutral | | |

---

## Summary

### Key Takeaways

1. **Code should support a question.** Each chart should answer something specific about the dataset.
2. **Chart choice changes the message.** Absolute totals, gaps, composition, and targets need different views.
3. **Styling should help interpretation.** Labels, color, and annotation should guide the reader.
4. **Real datasets need aggregation first.** The chart often depends on how you summarize the raw rows.
5. **A good chart is analytically useful.** It should help someone decide or investigate what to do next.

---

## Submission Checklist

- [ ] All 5 charts created in code and styled
- [ ] Real dataset aggregated appropriately for each question
- [ ] 10 files exported (5 PNG + 5 SVG)
- [ ] Critical analysis questions answered
- [ ] Source attribution on all charts

---

*End of Workshop*
